# Create Evaluation Tables

In [40]:
import re
from pathlib import Path

import pandas as pd

from hiatus.config import metrics_dir, tables_dir, metrics

In [55]:
def extract_dataset_name(path: str) -> str:
    """Extract dataset name from filename."""
    fname = Path(path).name

    # Pattern 1: perGenre-HRS3.302_...
    m = re.search(r"perGenre-(HRS[0-9.]+)", fname)
    if m:
        return m.group(1)

    # Pattern 2: zh_medium_crossGenre_...
    m = re.match(r"([a-zA-Z0-9_]+)_prompt", fname)
    if m:
        return m.group(1)

    return fname  # fallback


def build_metric_table(file_list, privacy_files=None, index="documentID", columns=None):
    """
    Build dataset-by-metric table.
    - file_list: JSONL files for sense_scores
    - privacy_files: CSV files for privacy_consistency (first column)
    """
    tables = {}

    # ------------------ SENSE SCORES ------------------
    for f in file_list:
        dataset = extract_dataset_name(f)
        df = pd.read_json(f, lines=True)

        df = df.drop(columns=index, errors="ignore")

        selected_cols = df.columns if columns is None else [c for c in columns if c in df.columns]
        df = df[selected_cols]

        # Apply scaling
        for col in df.columns:
            if col in metrics and "scale" in metrics[col]:
                df[col] = df[col] * metrics[col]["scale"]

        means = df.mean(axis=0)
        means.index = [metrics.get(col, {}).get("title", col) for col in means.index]

        tables[dataset] = means

    table = pd.DataFrame(tables)

    # --- PRIVACY SCORES ---
    if privacy_files:
        for f in privacy_files:
            dataset = extract_dataset_name(f)
            try:
                dfp = pd.read_csv(f)
            except Exception as e:
                print(f"[WARN] Could not read privacy CSV: {f}\n  {e}")
                continue

            dfp = dfp.drop(columns=index, errors="ignore")

            if dfp.shape[0] == 0:
                continue

            # Take all columns (not just first)
            for col in dfp.columns:
                mean_val = dfp[col].mean()
                title = metrics.get(col, {}).get("title", col)
                series = pd.Series({title: mean_val})
                table = table.combine_first(series.to_frame(dataset))

    return table


def build_model_score_table(metrics_path_dict, index="documentID", columns=None):
    """
    Build model-by-metric table, averaging across all datasets for each model.
    Includes sense_scores (JSONL) and privacy_consistency (CSV, first column).
    """
    model_summaries = {}

    for model_key, info in metrics_path_dict.items():
        sense_files = info["sense_scores"]
        privacy_files = info["privacy_consistency"]

        # ------------------ SENSE SCORES ------------------
        sense_dfs = []
        for f in sense_files:
            df = pd.read_json(f, lines=True)
            df = df.drop(columns=index, errors="ignore")

            selected_cols = df.columns if columns is None else [c for c in columns if c in df.columns]
            df = df[selected_cols]

            for col in df.columns:
                if col in metrics and "scale" in metrics[col]:
                    df[col] = df[col] * metrics[col]["scale"]

            sense_dfs.append(df)

        if sense_dfs:
            merged = pd.concat(sense_dfs, axis=0)
            sense_means = merged.mean(axis=0)
            sense_means.index = [metrics.get(col, {}).get("title", col) for col in sense_means.index]
        else:
            sense_means = pd.Series({})

        # ------------------ PRIVACY SCORES ------------------
        privacy_values = []
        privacy_metric_name = None

        for f in privacy_files:
            try:
                dfp = pd.read_csv(f)
            except Exception as e:
                print(f"[WARN] Could not read privacy CSV: {f}\n  {e}")
                continue

            dfp = dfp.drop(columns=index, errors="ignore")
            if dfp.shape[1] == 0:
                continue

            col = dfp.columns[0]
            privacy_values.append(dfp[col])
            if privacy_metric_name is None:
                privacy_metric_name = metrics.get(col, {}).get("title", col)

        if privacy_values:
            # Concatenate series from all files, then mean
            combined = pd.concat(privacy_values, axis=0)
            privacy_mean = pd.Series({privacy_metric_name: combined.mean()})
        else:
            privacy_mean = pd.Series({})

        # ------------------ MERGE ------------------
        full = pd.concat([sense_means, privacy_mean])
        model_summaries[info["label"]] = full

    return pd.DataFrame(model_summaries)

In [56]:
metrics_path = {
    "prompt-remix-zh0": {
        "sense_scores": [p.as_posix() for p in Path(metrics_dir, "zh").glob("*.jsonl")],
        "privacy_consistency": [p.as_posix() for p in Path(metrics_dir, "zh").glob("*.csv")],
        "color": "#027100",
        "label": "PromptRemix-ZH0",
        "hatch": False,
        "report": True,
    },
    "prompt-remix-ar0": {
        "sense_scores": [p.as_posix() for p in Path(metrics_dir, "ar").glob("*.jsonl")],
        "privacy_consistency": [p.as_posix() for p in Path(metrics_dir, "ar").glob("*.csv")],
        "color": "#DF6800",
        "label": "PromptRemix-AR0",
        "hatch": False,
        "report": True,
    },
    "prompt-remix-en0": {
        "sense_scores": [p.as_posix() for p in Path(metrics_dir, "en").glob("*.jsonl")],
        "privacy_consistency": [p.as_posix() for p in Path(metrics_dir, "en").glob("*.csv")],
        "color": "#004D80",
        "label": "PromptRemix-ENF",
        "hatch": True,
        "report": True,
    },
     "prompt-remix-ru0": {
        "sense_scores": [p.as_posix() for p in Path(metrics_dir, "ru").glob("*.jsonl")],
        "privacy_consistency": [p.as_posix() for p in Path(metrics_dir, "ru").glob("*.csv")],
        "color": "#800000",
        "label": "PromptRemix-RUF",
        "hatch": True,
        "report": True,
    },
}

## Model Specific

In [57]:
table_models = build_model_score_table(metrics_path)
print(table_models)


                              PromptRemix-ZH0  PromptRemix-AR0  \
Semantic Coverage (↑)                0.859733         0.413565   
Factuality (↑)                       0.882443         0.459464   
Informativeness (↑)                  0.851908         0.379022   
Relevance (↑)                        0.958588         0.507729   
Specificity (↑)                      0.794466         0.374448   
Correctness (↑)                      0.887405         0.485489   
Accuracy (↑)                         0.847328         0.420820   
Semantically Appropriate (↑)         0.823473         0.432019   
Consistency (↑)                      0.880916         0.471767   
Coherence (↑)                        0.879580         0.437697   
Fluency (↑)                          0.898855         0.499211   
Quality (↑)                          0.847137         0.417823   
Understandability (↑)                0.896183         0.485016   
Cosine Similarity (↑)                0.943985         0.899092   
Delta Equa

## Language-Specific

### ZH

In [59]:
zh_files = metrics_path["prompt-remix-zh0"]["sense_scores"]

metric_table = build_metric_table(zh_files, privacy_files=metrics_path["prompt-remix-zh0"]["privacy_consistency"])
print(metric_table)

                                           HRS3.301   HRS3.302   HRS3.303  \
Accuracy (↑)                               0.857143   0.831667   0.860000   
Coherence (↑)                              0.901429   0.898333   0.890000   
Consistency (↑)                            0.904286   0.881667   0.886667   
Correctness (↑)                            0.900000   0.883333   0.890000   
Cosine Similarity (↑)                      0.951475   0.943210   0.934828   
Delta Equal Error Rate                     0.498906   0.102099   0.272095   
Factuality (↑)                             0.880000   0.836667   0.890000   
Fluency (↑)                                0.947143   0.928333   0.933333   
Informativeness (↑)                        0.842857   0.838333   0.861667   
Quality (↑)                                0.870000   0.855000   0.865000   
Relevance (↑)                              0.961429   0.946667   0.966667   
Semantic Coverage (↑)                      0.861429   0.855000   0.885000   

### AR

In [63]:
ar_files = metrics_path["prompt-remix-ar0"]["sense_scores"]

metric_table = build_metric_table(ar_files, privacy_files=metrics_path["prompt-remix-ar0"]["privacy_consistency"])
print(metric_table)

                              ar_twitter_1  saudinewsnet
Accuracy (↑)                      0.424026      0.419792
Coherence (↑)                     0.437662      0.437708
Consistency (↑)                   0.462338      0.474792
Correctness (↑)                   0.462987      0.492708
Cosine Similarity (↑)             0.878510      0.905695
Delta Equal Error Rate           -0.149351      0.171875
Factuality (↑)                    0.439610      0.465833
Fluency (↑)                       0.462338      0.511042
Informativeness (↑)               0.412987      0.368125
Quality (↑)                       0.419481      0.417292
Relevance (↑)                     0.469481      0.520000
Semantic Coverage (↑)             0.418182      0.412083
Semantically Appropriate (↑)      0.410390      0.438958
Specificity (↑)                   0.396753      0.367292
Understandability (↑)             0.445455      0.497708


### EN

In [64]:
en_files = metrics_path["prompt-remix-en0"]["sense_scores"]

metric_table = build_metric_table(en_files,  privacy_files=metrics_path["prompt-remix-en0"]["privacy_consistency"])
print(metric_table)

                                HRS2.1    HRS2.2    HRS2.3    HRS2.4  \
Accuracy (↑)                  0.853247  0.881875  0.844444  0.907831   
Coherence (↑)                 0.857792  0.898125  0.870833  0.928916   
Consistency (↑)               0.883766  0.929375  0.872222  0.961446   
Correctness (↑)               0.900649  0.928125  0.879861  0.943373   
Cosine Similarity (↑)         0.936149  0.933030  0.951461  0.956798   
Delta Equal Error Rate        0.412798  0.325598  0.319810  0.204704   
Factuality (↑)                0.924026  0.938125  0.902083  0.951807   
Fluency (↑)                   0.857143  0.896875  0.889583  0.936145   
Informativeness (↑)           0.846753  0.884375  0.872222  0.909036   
Quality (↑)                   0.833117  0.868125  0.836806  0.915060   
Relevance (↑)                 0.971429  0.983125  0.976389  0.994578   
Semantic Coverage (↑)         0.858442  0.890625  0.868056  0.899398   
Semantically Appropriate (↑)  0.805195  0.855625  0.795833  0.91

### RU

In [66]:
ru_files = metrics_path["prompt-remix-ru0"]["sense_scores"]

metric_table = build_metric_table(ru_files,  privacy_files=metrics_path["prompt-remix-ru0"]["privacy_consistency"])
print(metric_table)

                              HRS2.101  HRS2.102  HRS2.103  HRS2.104  \
Accuracy (↑)                  0.833846  0.812500  0.834426  0.844079   
Coherence (↑)                 0.873077  0.843382  0.887705  0.862500   
Consistency (↑)               0.853846  0.843382  0.875410  0.873684   
Correctness (↑)               0.867692  0.860294  0.877049  0.891447   
Cosine Similarity (↑)         0.944722  0.957543  0.947310  0.946606   
Delta Equal Error Rate        0.185280  0.218621  0.341975  0.204879   
Factuality (↑)                0.880000  0.885294  0.865574  0.901316   
Fluency (↑)                   0.937692  0.875000  0.928689  0.913816   
Informativeness (↑)           0.845385  0.816912  0.827049  0.834211   
Quality (↑)                   0.846923  0.810294  0.845082  0.846053   
Relevance (↑)                 0.949231  0.930882  0.957377  0.950000   
Semantic Coverage (↑)         0.854615  0.816912  0.854098  0.855263   
Semantically Appropriate (↑)  0.816923  0.787500  0.825410  0.81